# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse_expressions/Conformance/Corpus.py`, mirrored statement for
statement in TypeScript) and commits its snapshots under `conformance/<implementation>/`. This suite checks this
implementation's files are current, that its JSON is byte-identical to every other implementation's, and that it reads
every other implementation's JSON and YAML back to the same expressions. Regenerate with
`uv run python -m mbse_expressions.Conformance.write`.

In [ ]:
from pathlib import Path

import yaml

from mbse_expressions import Expressions
from mbse_expressions.Conformance.Corpus import CASES, build
from mbse_expressions.Conformance.write import render
from mbse_schemas.Framework import JSON, Plain, Validators, YAML
from support import same_graph

ROOT = Path("../../conformance")
OWN = "python3"
OTHERS = sorted(p.name for p in ROOT.iterdir() if p.is_dir() and p.name != OWN)
assert OTHERS == ["typescript5"], OTHERS

corpus = build()
snapshots = {case: Plain.ToPlain.Reachable(schema, root) for case, (schema, root) in corpus.items()}

## CONF-01 · This implementation's committed files are current

In [ ]:
for name, text in render(corpus).items():
    assert (ROOT / OWN / name).read_text(encoding="utf-8") == text, f"{name} is stale: regenerate the corpus"

## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
for name in [OWN, *OTHERS]:
    assert sorted(p.name for p in (ROOT / name).iterdir()) == sorted(f"{c}.{e}" for c in CASES for e in ("json", "yaml")), name
for other in OTHERS:
    for case in CASES:
        mine = (ROOT / OWN / f"{case}.json").read_bytes()
        theirs = (ROOT / other / f"{case}.json").read_bytes()
        assert mine == theirs, f"{case}.json differs from {other}"

## CONF-03 · Every implementation's YAML reads back to exactly this implementation's snapshot, also under YAML 1.1

In [ ]:
for other in [OWN, *OTHERS]:
    for case in CASES:
        text = (ROOT / other / f"{case}.yaml").read_text(encoding="utf-8")
        assert YAML.loads(text) == snapshots[case], f"{other}/{case}.yaml"
        assert yaml.safe_load(text) == snapshots[case], f"{other}/{case}.yaml under YAML 1.1"

## CONF-04 · Every implementation's JSON and YAML deserialize with this implementation's builders

In [ ]:
validate = Validators.Validate(Expressions.Builders)
for other in [OWN, *OTHERS]:
    for case, (schema, _) in corpus.items():
        for suffix, decode in [("json", JSON.FromJSON(Expressions.Builders).Reachable),
                               ("yaml", YAML.FromYAML(Expressions.Builders).Reachable)]:
            restored = decode(schema, (ROOT / other / f"{case}.{suffix}").read_bytes())
            assert same_graph(Plain.ToPlain.Reachable(schema, restored), snapshots[case]), f"{other}/{case}.{suffix}"
            assert validate.Reachable(schema, restored) == [], (other, case, suffix)